In [47]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.tools import tool

import requests
import random

import os

In [48]:
load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT_EUS2')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_APIKEY_EUS2')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
LLM_DEPLOYMENT_NAME = os.getenv('LLM_DEPLOYMENT_NAME', os.getenv('MODEL_NAME'))


model = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=LLM_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0.2,
)

In [49]:
#tools

search_tool = DuckDuckGoSearchResults(region="us", safesearch="Moderate", max_results=5)


@tool("calculator")
def calculator(first_num:float,second_num:float,operation:str)->str:
    """Perform a basic arithmetic operation on two numbers.
    supported operations: add,sub,mul,div"""
    if operation == 'add':
        return str(first_num + second_num)
    elif operation == 'subtract':
        return str(first_num - second_num)
    elif operation == 'multiply':
        return str(first_num * second_num)
    elif operation == 'divide':
        if second_num == 0:
            return "Error: Division by zero"
        return str(first_num / second_num)
    else:
        return "Error: Invalid operation. Supported operations are add, subtract, multiply, divide."
    


@tool("joke_generator")

def joke_generator(user_input:str)->str:
    """Generate a joke based on the user's input."""
    prompt = f"Explain the joke {user_input} in detail"
    response = model.invoke(prompt).content
    return {
        'explaination': response,
    }
    

In [50]:
tools = [search_tool,calculator,joke_generator]

llm_with_tools = model.bind_tools(tools)

In [51]:
#state
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [52]:
def chat_node(state:ChatState)->ChatState:
    """LLM node thast may answer or request a tool call"""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages":[response]}
    
tool_node = ToolNode(tools)#execute tool calls

In [ ]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools",tool_node)


In [ ]:
graph.add_edge(START,"chat_node")
graph.add_conditional_edges("chat_node",tools_condition)
graph.add_edge("tools", "chat_node")
graph.add_edge("chat_node",END)

In [ ]:
checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)
initial_state = {"messages":[HumanMessage(content=" joke on pizza")]}
config1 = {"configurable": {"thread_id": "1"}}


result = workflow.invoke(initial_state,config=config1)

print(result['messages'][-1].content)


{"explaination": "“Pizza” by itself isn’t a joke, so I need the exact joke or meme you mean.\n\nPlease paste it, and I’ll explain:\n- the literal meaning\n- the wordplay/double meaning\n- why it’s funny\n- any cultural context\n\nIf you want, I can also explain common pizza jokes like:\n- “What type of person doesn’t like pizza? A weir-dough.”\n- “Little Caesars is called that because they only hire little people.”\n- “Pizza de resistance” / “piece-a” puns\n\nSend the exact joke and I’ll break it down in detail."}


Failed to batch ingest runs: langsmith.utils.LangSmithConnectionError: Connection error caused failure to POST https://api.smith.langchain.com/runs/batch in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/batch (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))"))
Content-Length: 5645
API Key: lsv2_********************************************1a
post: trace=0e6bdc97-3d85-4980-9d6f-693cbe5fdcfa,id=0e6bdc97-3d85-4980-9d6f-693cbe5fdcfa; trace=0e6bdc97-3d85-4980-9d6f-693cbe5fdcfa,id=911ce422-9bf1-4453-b161-b74746ad384d; trace=0e6bdc97-3d85-4980-9d6f-693cbe5fdcfa,id=98f9d388-3ec9-40ac-a5e3-b35417a8ce65
Failed to batch ingest runs: langsmith.utils.LangSmithConnectionError: Connection error caused failure to POST https://api.smith.langchain.com/runs/b

In [65]:
workflow.get_state(config1)


StateSnapshot(values={'messages': [HumanMessage(content=' joke on pizza', additional_kwargs={}, response_metadata={}, id='15e29faf-f02c-42ff-86db-02c6d8489a23'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Kyvb7pzqrWaVCRQzQowZ9ghl', 'function': {'arguments': '{"user_input":"pizza"}', 'name': 'joke_generator'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 225, 'total_tokens': 245, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DUVG00n0kPHBWGdikwg82gliklZOE', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 

In [63]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'messages': [HumanMessage(content='give me a joke on pizza', additional_kwargs={}, response_metadata={}, id='28e59e96-1e17-4859-96e2-9d06f1766909'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_KsBl9O5IbiAVGNseRSJhoVdY', 'function': {'arguments': '{"user_input":"pizza"}', 'name': 'joke_generator'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 228, 'total_tokens': 248, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DUVEni9CdF85IDt2jqZdBEqRkyTSY', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered'